# Module 4: Exploration, Exploitation, and Model-Free Methods

**Spinning Up in Active Inference** | Notebook 4 of 16

---

Every agent faces a fundamental dilemma: should it **exploit** what it already knows to collect reward, or **explore** uncertain parts of the environment that might yield better outcomes? This is the exploration-exploitation tradeoff, and how different frameworks handle it reveals deep differences in their design philosophy.

Model-free RL treats exploration as a *problem to be solved* — something that must be bolted onto the reward-maximization objective with mechanisms like epsilon-greedy or Boltzmann exploration. Active Inference dissolves the dilemma entirely: exploration (epistemic value) and exploitation (pragmatic value) are both components of a single objective function, the expected free energy.

In this module:
1. **Epsilon-greedy and softmax** exploration strategies
2. **SARSA vs Q-learning**: on-policy vs off-policy, and why SARSA is safer
3. **Exploration heatmaps**: visualizing how agents distribute their visits
4. **The AIF perspective**: how epistemic value provides principled exploration

**Prerequisites:** Modules 1-3.  
**Time:** ~60 minutes.

## 4.1 The Exploration-Exploitation Dilemma

Imagine you have discovered a restaurant that you enjoy. Every evening you face a choice:
- **Exploit:** Go to the known-good restaurant (guaranteed decent meal)
- **Explore:** Try a new restaurant (might be terrible, might be amazing)

Pure exploitation means you never discover better options. Pure exploration means you never benefit from what you have learned. The optimal strategy depends on:
- How much you have already explored
- How much time you have left (finite horizon vs infinite)
- How uncertain you are about unexplored options

In RL, this is not just a philosophical question — it has real computational consequences. An agent that exploits too early converges to a **suboptimal policy**. An agent that explores too much wastes time on low-reward states.

Let us examine the standard approaches.

## 4.2 Epsilon-Greedy Exploration

The simplest exploration strategy: with probability $\epsilon$, take a random action; with probability $1 - \epsilon$, take the greedy (highest Q-value) action.

$$\pi(a|s) = \begin{cases} 1 - \epsilon + \epsilon/|\mathcal{A}| & \text{if } a = \arg\max_a Q(s,a) \\ \epsilon/|\mathcal{A}| & \text{otherwise} \end{cases}$$

**Pros:** Simple, guaranteed to explore all states eventually.  
**Cons:** Exploration is **undirected** — it does not distinguish between genuinely uncertain states and well-understood ones. It explores with equal probability everywhere, even in states where it already knows the optimal action.

## 4.3 Softmax (Boltzmann) Exploration

A more graded approach: actions with higher Q-values are more likely, but all actions have nonzero probability:

$$\pi(a|s) = \frac{\exp(\beta \cdot Q(s,a))}{\sum_{a'} \exp(\beta \cdot Q(s,a'))}$$

The **inverse temperature** $\beta$ controls exploration:
- $\beta \to 0$: uniform random (pure exploration)
- $\beta \to \infty$: deterministic greedy (pure exploitation)

**Pros:** Exploration is proportional to action values — promising actions get more attention.  
**Cons:** Still does not account for *uncertainty* about action values. High-Q actions might just be well-estimated, not actually good.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate
from neuronav.agents.td_agents import TDQ, SARSA

import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_value_heatmap, plot_learning_curve, plot_agent_trajectory,
    dual_perspective_box, COLORS
)

np.random.seed(42)
print("Imports successful!")

In [ ]:
# ── Implement epsilon-greedy and softmax from scratch ─────────────────────────

def epsilon_greedy(q_values, epsilon=0.1):
    """Epsilon-greedy action selection.
    
    Args:
        q_values: Q-values for current state, shape (n_actions,)
        epsilon: Exploration probability
    
    Returns:
        Selected action index
    """
    if np.random.random() < epsilon:
        return np.random.randint(len(q_values))
    else:
        return np.argmax(q_values)

def softmax_action(q_values, beta=1.0):
    """Softmax (Boltzmann) action selection.
    
    Args:
        q_values: Q-values for current state, shape (n_actions,)
        beta: Inverse temperature (higher = more exploitative)
    
    Returns:
        Selected action index
    """
    # Subtract max for numerical stability
    logits = beta * (q_values - np.max(q_values))
    probs = np.exp(logits) / np.sum(np.exp(logits))
    return np.random.choice(len(q_values), p=probs)

# Demonstrate with some Q-values
q_example = np.array([0.5, 2.0, 1.5, 0.8])
action_names = ['Up', 'Down', 'Left', 'Right']

print("Q-values:", dict(zip(action_names, q_example)))
print()

# Show action probabilities for different strategies
n_samples = 10000

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (strategy, params, label) in zip(axes, [
    (epsilon_greedy, {'epsilon': 0.1}, '\u03B5-greedy (\u03B5=0.1)'),
    (softmax_action, {'beta': 1.0}, 'Softmax (\u03B2=1.0)'),
    (softmax_action, {'beta': 10.0}, 'Softmax (\u03B2=10.0)'),
]):
    counts = np.zeros(4)
    for _ in range(n_samples):
        a = strategy(q_example, **params)
        counts[a] += 1
    probs = counts / n_samples
    
    colors = [COLORS['highlight'] if p == max(probs) else COLORS['neutral'] for p in probs]
    ax.bar(action_names, probs, color=colors, alpha=0.85)
    ax.set_ylim(0, 1)
    ax.set_ylabel('P(action)')
    ax.set_title(label, fontsize=12)
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f'{p:.3f}', ha='center', fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Action Selection Strategies (Q = [0.5, 2.0, 1.5, 0.8])', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4.4 SARSA vs Q-Learning: On-Policy vs Off-Policy

**Q-learning** (off-policy) uses the maximum Q-value for the update, regardless of which action was actually taken:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

**SARSA** (on-policy) uses the Q-value of the action that was actually selected next:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma Q(s', a') - Q(s, a) \right]$$

where $a'$ is the action actually taken in state $s'$.

The classic demonstration of their difference is **cliff-walking**: Q-learning learns the optimal (cliff-edge) path because it evaluates the greedy policy, but SARSA learns a safer path because it accounts for its own exploratory actions that might send it over the edge.

In [ ]:
# ── SARSA vs Q-Learning on Four Rooms ────────────────────────────────────────

env = GridEnv(GridSize.small, obs_type=GridObservation.index, 
              template=GridTemplate.four_rooms)
state_size = env.grid_size ** 2
action_size = 4

# Train both agents
n_episodes = 500

q_agent = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
              poltype="softmax", beta=5.0)
sarsa_agent = SARSA(state_size, action_size, lr=0.1, gamma=0.99,
                    poltype="softmax", beta=5.0)

q_rewards = []
sarsa_rewards = []
q_visits = np.zeros(state_size)  # count state visits
sarsa_visits = np.zeros(state_size)

for ep in range(n_episodes):
    # Q-learning
    obs = env.reset()
    done = False
    total_r = 0
    steps = 0
    while not done and steps < 500:
        action = q_agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        q_agent.update((obs, action, next_obs, reward, done))
        q_visits[obs] += 1
        obs = next_obs
        total_r += reward
        steps += 1
    q_rewards.append(total_r)

    # SARSA
    obs = env.reset()
    action = sarsa_agent.sample_action(obs)
    done = False
    total_r = 0
    steps = 0
    while not done and steps < 500:
        next_obs, reward, done, info = env.step(action)
        next_action = sarsa_agent.sample_action(next_obs)
        sarsa_agent.update((obs, action, next_obs, reward, done))
        sarsa_visits[obs] += 1
        obs = next_obs
        action = next_action
        total_r += reward
        steps += 1
    sarsa_rewards.append(total_r)

print(f"Q-learning mean reward (last 50): {np.mean(q_rewards[-50:]):.3f}")
print(f"SARSA mean reward (last 50): {np.mean(sarsa_rewards[-50:]):.3f}")

In [ ]:
# ── Compare learning curves ──────────────────────────────────────────────────

plot_learning_curve(
    {"Q-Learning (off-policy)": q_rewards, "SARSA (on-policy)": sarsa_rewards},
    title="Q-Learning vs SARSA: Learning Curves (500 episodes)",
    ylabel="Episode Reward",
    window=20
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualize exploration patterns: state visit heatmaps ─────────────────────
# These heatmaps reveal HOW each agent explores the environment.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

plot_value_heatmap(
    np.log1p(q_visits), env.grid_size,
    title="Q-Learning: State Visit Counts (log scale)\nExploration pattern",
    cmap="YlOrRd",
    show_values=False,
    goal_pos=tuple(env.goal_pos),
    ax=ax1
)

plot_value_heatmap(
    np.log1p(sarsa_visits), env.grid_size,
    title="SARSA: State Visit Counts (log scale)\nExploration pattern",
    cmap="YlOrRd",
    show_values=False,
    goal_pos=tuple(env.goal_pos),
    ax=ax2
)

plt.tight_layout()
plt.show()
print("Compare the exploration patterns: which agent visits more of the state space?")

In [ ]:
# ── Compare value functions ──────────────────────────────────────────────────

V_q = np.max(q_agent.q_table, axis=1)
V_sarsa = np.max(sarsa_agent.q_table, axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

plot_value_heatmap(
    V_q, env.grid_size,
    title="Q-Learning: V(s) = max_a Q(s,a)",
    cmap="inferno",
    goal_pos=tuple(env.goal_pos),
    show_values=False,
    ax=ax1
)

plot_value_heatmap(
    V_sarsa, env.grid_size,
    title="SARSA: V(s) = max_a Q(s,a)",
    cmap="inferno",
    goal_pos=tuple(env.goal_pos),
    show_values=False,
    ax=ax2
)

plt.tight_layout()
plt.show()

## 4.5 The Effect of Exploration Parameters

How do exploration parameters affect performance? Let us systematically vary epsilon and beta to see their impact.

In [ ]:
# ── Effect of epsilon on Q-learning ──────────────────────────────────────────

epsilons = [0.01, 0.1, 0.3]
eps_curves = {}

for eps in epsilons:
    env_eps = GridEnv(GridSize.small, obs_type=GridObservation.index, 
                      template=GridTemplate.four_rooms)
    agent_eps = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
                    poltype="epsilon", epsilon=eps)
    rewards_eps = []
    for ep in range(500):
        obs = env_eps.reset()
        done = False
        total_r = 0
        steps = 0
        while not done and steps < 500:
            action = agent_eps.sample_action(obs)
            next_obs, reward, done, info = env_eps.step(action)
            agent_eps.update((obs, action, next_obs, reward, done))
            obs = next_obs
            total_r += reward
            steps += 1
        rewards_eps.append(total_r)
    eps_curves[f"\u03B5 = {eps}"] = rewards_eps

plot_learning_curve(
    eps_curves,
    title="Effect of \u03B5 on Q-Learning Performance",
    ylabel="Episode Reward",
    window=20
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Effect of beta (softmax temperature) on exploration ──────────────────────

betas = [0.5, 2.0, 10.0]
beta_curves = {}

for beta in betas:
    env_beta = GridEnv(GridSize.small, obs_type=GridObservation.index, 
                       template=GridTemplate.four_rooms)
    agent_beta = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
                     poltype="softmax", beta=beta)
    rewards_beta = []
    for ep in range(500):
        obs = env_beta.reset()
        done = False
        total_r = 0
        steps = 0
        while not done and steps < 500:
            action = agent_beta.sample_action(obs)
            next_obs, reward, done, info = env_beta.step(action)
            agent_beta.update((obs, action, next_obs, reward, done))
            obs = next_obs
            total_r += reward
            steps += 1
        rewards_beta.append(total_r)
    beta_curves[f"\u03B2 = {beta}"] = rewards_beta

plot_learning_curve(
    beta_curves,
    title="Effect of \u03B2 (Softmax Temperature) on Q-Learning Performance",
    ylabel="Episode Reward",
    window=20
)
plt.tight_layout()
plt.show()

print("Low \u03B2 (0.5): Nearly random exploration, slow learning.")
print("Medium \u03B2 (2.0): Balanced exploration-exploitation.")
print("High \u03B2 (10.0): Mostly greedy, risks getting stuck in local optima early.")

In [ ]:
# ── Dual Perspective: Exploration in RL vs. AIF ──────────────────────────────

dual_perspective_box(
    rl_text=(
        "RL agents need explicit exploration mechanisms bolted onto the reward-maximization "
        "objective. \u03B5-greedy explores randomly and uniformly — it does not distinguish "
        "between genuinely uncertain states and well-understood ones. Softmax is better but "
        "still does not account for epistemic uncertainty. UCB adds optimism in the face of "
        "uncertainty, but is limited to bandits. The exploration-exploitation tradeoff remains "
        "an unsolved problem in general RL."
    ),
    aif_text=(
        "Active Inference dissolves the exploration-exploitation dilemma. The Expected Free "
        "Energy G(\u03C0) = -pragmatic - epistemic naturally combines reward-seeking (pragmatic: "
        "'will this policy reach my preferred observations?') with information-seeking "
        "(epistemic: 'will this policy reduce my uncertainty?'). An AIF agent explores "
        "BECAUSE reducing uncertainty IS part of the objective — no separate mechanism needed. "
        "Early on, epistemic value dominates (explore). Later, pragmatic value dominates (exploit)."
    ),
    title="Exploration: Bolted-On (RL) vs. Intrinsic (Active Inference)"
)

## 4.6 Upper Confidence Bound (UCB)

A more principled approach to exploration is **UCB** (Auer et al., 2002), which selects actions based on:

$$a^* = \arg\max_a \left[ Q(s,a) + c \sqrt{\frac{\ln N(s)}{N(s,a)}} \right]$$

where $N(s)$ is the number of visits to state $s$, and $N(s,a)$ is the number of times action $a$ was taken in state $s$. The second term is an **exploration bonus** that:
- Grows with visits to the state (more opportunities to explore)
- Shrinks with visits to the specific action (already explored)
- Is controlled by parameter $c$

This embodies **optimism in the face of uncertainty**: untried actions get a bonus that decays with experience. This is closer in spirit to Active Inference's epistemic value, which also drives exploration toward uncertain regions of state space.

In [ ]:
# ── Exercise 1: Implement UCB action selection ───────────────────────────────

def ucb_action(q_values, visit_counts_state, visit_counts_sa, c=2.0):
    """UCB action selection.
    
    Args:
        q_values: Q-values for current state, shape (n_actions,)
        visit_counts_state: Total visits to this state
        visit_counts_sa: Visits for each action in this state, shape (n_actions,)
        c: Exploration constant
    
    Returns:
        Selected action index
    """
    n_actions = len(q_values)
    # Handle unvisited actions: try them first
    unvisited = np.where(visit_counts_sa == 0)[0]
    if len(unvisited) > 0:
        return np.random.choice(unvisited)
    
    # UCB formula
    ucb_values = q_values + c * np.sqrt(np.log(visit_counts_state) / visit_counts_sa)
    return np.argmax(ucb_values)

# Train a UCB-based Q-learning agent
env_ucb = GridEnv(GridSize.small, obs_type=GridObservation.index, 
                  template=GridTemplate.four_rooms)

q_table_ucb = np.zeros((state_size, action_size))
state_visits = np.zeros(state_size)
sa_visits = np.zeros((state_size, action_size))
lr = 0.1
gamma = 0.99

ucb_rewards = []
for ep in range(500):
    obs = env_ucb.reset()
    done = False
    total_r = 0
    steps = 0
    while not done and steps < 500:
        action = ucb_action(q_table_ucb[obs], state_visits[obs], sa_visits[obs])
        next_obs, reward, done, info = env_ucb.step(action)
        
        # Q-learning update
        td_target = reward + gamma * np.max(q_table_ucb[next_obs]) * (1 - done)
        q_table_ucb[obs, action] += lr * (td_target - q_table_ucb[obs, action])
        
        state_visits[obs] += 1
        sa_visits[obs, action] += 1
        obs = next_obs
        total_r += reward
        steps += 1
    ucb_rewards.append(total_r)

# Compare all three exploration strategies
plot_learning_curve(
    {
        "\u03B5-greedy (\u03B5=0.1)": eps_curves["\u03B5 = 0.1"],
        "Softmax (\u03B2=2.0)": beta_curves["\u03B2 = 2.0"],
        "Q-Learning + UCB": ucb_rewards,
    },
    title="Exploration Strategy Comparison",
    ylabel="Episode Reward",
    window=20
)
plt.tight_layout()
plt.show()

## 4.7 Exercises

### Exercise 2: Design an Environment Where SARSA Outperforms Q-Learning

SARSA's conservative behavior (accounting for its own exploration) is advantageous when exploration can be costly. Think about environments where random exploration leads to bad states (negative rewards, long detours).

In [ ]:
# ── Exercise 2: Environment where SARSA's conservatism helps ──────────────────
# We simulate a "cliff-like" scenario by penalizing certain states.
# The agent must navigate near dangerous states to reach the goal efficiently.

# Use the standard Four Rooms but track online performance (not just final policy)
# SARSA's advantage shows in the CUMULATIVE reward during training,
# not just the final policy. SARSA avoids costly exploratory mistakes.

env_cliff = GridEnv(GridSize.small, obs_type=GridObservation.index, 
                    template=GridTemplate.four_rooms)

# Train both agents and track cumulative reward during training
q_agent_2 = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
                poltype="softmax", beta=3.0)  # lower beta = more exploration = more danger
sarsa_agent_2 = SARSA(state_size, action_size, lr=0.1, gamma=0.99,
                      poltype="softmax", beta=3.0)

q_cumulative = []
sarsa_cumulative = []
q_running = 0
sarsa_running = 0

for ep in range(300):
    # Q-learning
    obs = env_cliff.reset()
    done = False
    steps = 0
    while not done and steps < 500:
        action = q_agent_2.sample_action(obs)
        next_obs, reward, done, info = env_cliff.step(action)
        q_agent_2.update((obs, action, next_obs, reward, done))
        q_running += reward
        obs = next_obs
        steps += 1
    q_cumulative.append(q_running)

    # SARSA
    obs = env_cliff.reset()
    action = sarsa_agent_2.sample_action(obs)
    done = False
    steps = 0
    while not done and steps < 500:
        next_obs, reward, done, info = env_cliff.step(action)
        next_action = sarsa_agent_2.sample_action(next_obs)
        sarsa_agent_2.update((obs, action, next_obs, reward, done))
        sarsa_running += reward
        obs = next_obs
        action = next_action
        steps += 1
    sarsa_cumulative.append(sarsa_running)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(q_cumulative, label='Q-Learning (cumulative)', color=COLORS['rl'], linewidth=2)
ax.plot(sarsa_cumulative, label='SARSA (cumulative)', color=COLORS['aif'], linewidth=2)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Cumulative Reward', fontsize=12)
ax.set_title('Cumulative Reward During Training\n'
             'SARSA\'s conservatism reduces costly exploration mistakes', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 3: What Would an AIF Agent Do?

Consider the cliff-walking problem. An RL agent with epsilon-greedy exploration might randomly step off the cliff. An AIF agent would behave differently:

1. **Early on** (high uncertainty): The agent's epistemic value dominates. It explores to build a model of the environment, including learning where the cliff is.

2. **After learning the cliff**: The agent's model predicts that cliff states lead to highly surprising (low probability) observations. Its pragmatic value drives it to avoid these states.

3. **Crucially**: The AIF agent does not need epsilon-greedy to explore. Its uncertainty about unvisited states provides *intrinsic motivation* to explore them. And once it has explored the cliff edge, its model ensures it avoids it — not because of a negative reward signal, but because falling off the cliff would be *surprising*.

This is the fundamental difference: RL agents need external exploration mechanisms. AIF agents explore because uncertainty reduction is part of the objective. We will implement this in Module 7 when we build our first Active Inference agent.

In [ ]:
# ── Preview: EFE decomposes into pragmatic + epistemic ───────────────────────
# This is a conceptual preview of what we will build in Module 7.

from alf.free_energy import expected_free_energy_decomposed

# Simple example: 3 states, 2 actions
A = np.array([
    [0.9, 0.1, 0.1],   # Observation 0 is likely in state 0
    [0.1, 0.9, 0.1],   # Observation 1 is likely in state 1
    [0.0, 0.0, 0.8],   # Observation 2 is likely in state 2
])
# Normalize columns
A = A / A.sum(axis=0, keepdims=True)

B = np.zeros((3, 3, 2))
# Action 0: move toward state 0 (known, rewarding)
B[:, :, 0] = [[0.8, 0.3, 0.1],
               [0.1, 0.5, 0.2],
               [0.1, 0.2, 0.7]]
# Action 1: move toward state 2 (uncertain, informative)
B[:, :, 1] = [[0.2, 0.1, 0.1],
               [0.3, 0.3, 0.2],
               [0.5, 0.6, 0.7]]

# Preferences: prefer observation 0 ("reward")
C = np.array([2.0, 0.0, -1.0])  # log-preferences

# Current belief: uncertain between states 1 and 2
beliefs = np.array([0.1, 0.45, 0.45])

# Compute EFE for both actions
results = []
for a in range(2):
    efe = expected_free_energy_decomposed(A, B, C, beliefs, a)
    results.append(efe)
    action_name = "Exploit (go to known reward)" if a == 0 else "Explore (go to uncertain state)"
    print(f"Action {a} ({action_name}):")
    print(f"  Pragmatic value: {efe.pragmatic:+.3f}")
    print(f"  Epistemic value: {efe.epistemic:+.3f}")
    print(f"  Total G(a):      {efe.G_total:+.3f}")
    print()

In [ ]:
# ── Visualize the EFE decomposition ──────────────────────────────────────────
from utils.plotting import plot_efe_decomposition

pragmatic_values = [r.pragmatic for r in results]
epistemic_values = [r.epistemic for r in results]

plot_efe_decomposition(
    pragmatic=pragmatic_values,
    epistemic=epistemic_values,
    action_names=["Exploit\n(known reward)", "Explore\n(uncertain state)"],
    title="Expected Free Energy Decomposition\n"
          "AIF naturally balances exploration and exploitation"
)
plt.tight_layout()
plt.show()

print("The action with the LOWEST total G (most negative) is preferred.")
print("Notice how epistemic value can make exploration the preferred action,")
print("even when the exploit action has higher pragmatic value.")

## 4.8 Summary: What We Have Learned in Part I

Over these four modules, we have built up the model-free RL toolkit:

| Module | Key Concept | Algorithm | AIF Parallel |
|--------|-------------|-----------|-------------|
| 1. Behaving Animal | S-R learning vs. cognitive maps | Random agent, TDQ | Generative models |
| 2. Bellman & Value | Recursive value computation | Value iteration | Expected free energy |
| 3. TD Learning | Prediction errors drive learning | TD(0), SR | Surprise minimization |
| 4. Exploration | Balancing explore vs. exploit | epsilon-greedy, softmax, UCB | Epistemic value |

In **Part II** (Modules 5-8), we cross the bridge to Active Inference:
- Module 5: **Model-Based RL** — learning a model of the world (Dyna, MCTS)
- Module 6: **Generative Models** — the statistical foundation of AIF
- Module 7: **Your First Active Inference Agent** — building with PGMax
- Module 8: **Expected Free Energy** — the unifying objective

**Next:** [Module 5: Model-Based RL and Planning](05_model_based_rl.ipynb)

---

## Further Reading

1. **Sutton, R.S. & Barto, A.G. (2018).** *Reinforcement Learning: An Introduction* (2nd ed.), Chapter 6 (TD Learning), Chapter 2 (Bandits/Exploration). The standard reference for all algorithms covered here.

2. **Auer, P., Cesa-Bianchi, N. & Fischer, P. (2002).** *Finite-time Analysis of the Multiarmed Bandit Problem.* Machine Learning, 47(2-3), 235-256. The foundational UCB paper.

3. **Schwartenbeck, P. et al. (2019).** *Computational Mechanisms of Curiosity and Goal-Directed Exploration.* eLife. Shows how Active Inference accounts for curious, exploratory behavior through epistemic value.

4. **Friston, K.J. et al. (2015).** *Active Inference and Epistemic Value.* Cognitive Neuroscience, 6(4), 187-214. The definitive paper on how AIF dissolves the exploration-exploitation dilemma.

5. **Singh, S., Lewis, R.L., Barto, A.G. & Sorg, J. (2010).** *Intrinsically Motivated Reinforcement Learning: An Evolutionary Perspective.* IEEE TAMD. An RL perspective on intrinsic motivation that connects to AIF's epistemic drive.